### Random Undersampling 

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from collections import Counter

input_dir  = './final_split_data/train_set.pkl'
output_dir = './mvts_rus'  # overwrite in same folder
os.makedirs(output_dir, exist_ok=True)

# Load data
# with open(f'{input_dir}/X.pkl', 'rb') as f:
    # X = pickle.load(f)

with open(f'{input_dir}', 'rb') as f:
    val_bundle = pickle.load(f)
X = val_bundle['X']
y = val_bundle['y']

# y = pd.read_csv(f'{input_dir}/y.csv', header=None).values.flatten().astype(int)

print(f"Before undersampling:")
for label, count in sorted(Counter(y).items()):
    print(f"  Class {label}: {count:,}")
print(f"  Total: {len(y):,}")

# ── Random undersample class 0 (keep 20%) ────────────────────────────────────
np.random.seed(42)

class0_idx   = np.where(y == 0)[0]
other_idx    = np.where(y != 0)[0]

n_keep       = int(len(class0_idx) * 0.2)   # keep 20% → remove 80%
class0_keep  = np.random.choice(class0_idx, size=n_keep, replace=False)

final_idx    = np.concatenate([class0_keep, other_idx])
final_idx    = np.sort(final_idx)             # preserve original row order

X_resampled  = X[final_idx]
y_resampled  = y[final_idx]

print(f"\nAfter undersampling:")
for label, count in sorted(Counter(y_resampled).items()):
    print(f"  Class {label}: {count:,}")
print(f"  Total: {len(y_resampled):,}")

# ── Save ─────────────────────────────────────────────────────────────────────
with open(f'{output_dir}/X.pkl', 'wb') as f:
    pickle.dump(X_resampled, f)

pd.Series(y_resampled).to_csv(f'{output_dir}/y.csv', index=False, header=False)

print(f"\nSaved X → {output_dir}/X.pkl  shape={X_resampled.shape}")
print(f"Saved y → {output_dir}/y.csv")
print("\nAll done!")

### Class distribution after RUS 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import os

input_dir  = './mvts_rus'
output_dir = './plots'
os.makedirs(output_dir, exist_ok=True)

y = pd.read_csv(f'{input_dir}/y.csv', header=None).values.flatten().astype(int)

label_names  = {0: 'NSEP', 1: 'SEP'}
label_colors = {
    0: '#94A3B8',  # cool slate
    1: '#F43F5E',  # rose
    2: '#10B981',  # emerald
    3: '#F59E0B',  # amber
    4: '#8B5CF6',  # violet
}

unique, counts = np.unique(y, return_counts=True)
total = counts.sum()

df_plot = pd.DataFrame({
    'Class'  : [label_names[i] for i in unique],
    'Samples': counts,
    'Color'  : [label_colors[i] for i in unique]
})

# ── Canvas ───────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', context='talk', font='DejaVu Sans')
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('#F8FAFC')
ax.set_facecolor('#F1F5F9')

# ── Bars ─────────────────────────────────────────────────────────────────────
sns.barplot(
    data=df_plot,
    x='Class', y='Samples',
    palette=df_plot['Color'].tolist(),
    edgecolor='white', linewidth=1.5,
    ax=ax, width=0.55
)

# Soft shadow behind each bar
for bar, label_id in zip(ax.patches, unique):
    cx = bar.get_x() + bar.get_width() / 2
    for alpha, extra_w in [(0.10, 0.30), (0.05, 0.55)]:
        ax.bar(cx, bar.get_height(),
               width=bar.get_width() + extra_w,
               color=label_colors[label_id],
               alpha=alpha, zorder=0, edgecolor='none')

# ── Labels on bars ───────────────────────────────────────────────────────────
for bar, count, label_id in zip(ax.patches, counts, unique):
    pct = 100 * count / total
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.004,
        f'{count:,}',
        ha='center', va='bottom',
        fontsize=13, fontweight='bold',
        color=label_colors[label_id],
    )
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.004 + counts.max() * 0.045,
        f'{pct:.1f}%',
        ha='center', va='bottom',
        fontsize=10, color='#64748B',
    )

# ── Grid & spines ────────────────────────────────────────────────────────────
ax.yaxis.grid(True, color='#CBD5E1', linewidth=0.8, linestyle='--')
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_color('#E2E8F0')

# ── Title & axis labels ──────────────────────────────────────────────────────
ax.set_title('Class Distribution', fontsize=22, fontweight='bold',
             color='#1E293B', pad=20)
ax.set_xlabel('Class', fontsize=14, color='#475569', labelpad=12)
ax.set_ylabel('Number of Samples', fontsize=14, color='#475569', labelpad=12)
ax.tick_params(colors='#475569', labelsize=12)
ax.set_ylim(0, counts.max() * 1.22)

# ── Total samples badge ──────────────────────────────────────────────────────
ax.text(0.99, 0.97, f'Total  {total:,} samples',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=11, color='#475569', fontstyle='italic',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                  edgecolor='#CBD5E1', linewidth=1.2))

plt.tight_layout()
plt.savefig(f'{output_dir}/class_distribution_rus.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()
print(f"Saved → {output_dir}/class_distribution_rus.png")

### tSNE plots of classes using MVTS and RUS

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os

input_dir  = './mvts_rus'
output_dir = './plots'
os.makedirs(output_dir, exist_ok=True)

# Load data
with open(f'{input_dir}/X.pkl', 'rb') as f:
    X = pickle.load(f)

y = pd.read_csv(f'{input_dir}/y.csv', header=None).values.flatten().astype(int)

# Flatten (num_samples, 288, 10) → (num_samples, 2880)
X_flat = X.reshape(X.shape[0], -1)

label_names  = {0: 'NSEP', 1: 'gt10', 2: 'gt30', 3: 'gt60', 4: 'gt100'}
label_colors = {
    0: '#94A3B8',  # cool slate
    1: '#F43F5E',  # rose
    2: '#10B981',  # emerald
    3: '#F59E0B',  # amber
    4: '#8B5CF6',  # violet
}
DOT_SMALL = 8
DOT_BIG   = 100

def plot_embedding(X_2d, y, title, filename, xlabel, ylabel):
    sns.set_theme(style='whitegrid', context='talk', font='DejaVu Sans')
    fig, ax = plt.subplots(figsize=(11, 8))
    fig.patch.set_facecolor('#F8FAFC')
    ax.set_facecolor('#F1F5F9')

    # Draw NSEP first so minority classes render on top
    for label_id in sorted(label_names.keys()):
        mask = y == label_id
        if mask.sum() == 0:
            continue

        is_nsep = label_id == 0
        ax.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            c=label_colors[label_id],
            s=DOT_SMALL if is_nsep else DOT_BIG,
            label=f"{label_names[label_id]}  (n={mask.sum():,})",
            alpha=0.4 if is_nsep else 0.9,
            edgecolors='none' if is_nsep else 'white',
            linewidths=0 if is_nsep else 0.5,
            zorder=1 if is_nsep else 2
        )

    ax.set_title(title, fontsize=20, fontweight='bold', color='#1E293B', pad=16)
    ax.set_xlabel(xlabel, fontsize=13, color='#475569', labelpad=10)
    ax.set_ylabel(ylabel, fontsize=13, color='#475569', labelpad=10)
    ax.tick_params(colors='#475569', labelsize=11)

    for spine in ax.spines.values():
        spine.set_color('#E2E8F0')
    ax.yaxis.grid(True, color='#CBD5E1', linewidth=0.6, linestyle='--')
    ax.xaxis.grid(True, color='#CBD5E1', linewidth=0.6, linestyle='--')

    legend = ax.legend(
        title='Class', title_fontsize=11,
        fontsize=10, loc='best',
        framealpha=0.95, edgecolor='#CBD5E1',
        facecolor='white'
    )
    legend.get_title().set_color('#1E293B')

    plt.tight_layout()
    plt.savefig(f'{output_dir}/{filename}', dpi=150,
                bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close()
    print(f"Saved → {output_dir}/{filename}")

# ── PCA ───────────────────────────────────────────────────────────────────────
print("Running PCA...")
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_flat)
var   = pca.explained_variance_ratio_ * 100
print(f"  Explained variance: PC1={var[0]:.1f}%  PC2={var[1]:.1f}%  Total={sum(var):.1f}%")

plot_embedding(
    X_pca, y,
    title=f'PCA — Multivariate Dataset  (PC1={var[0]:.1f}%  PC2={var[1]:.1f}%)',
    filename='pca_rus.png',   # ← changed
    xlabel=f'PC1  ({var[0]:.1f}% variance)',
    ylabel=f'PC2  ({var[1]:.1f}% variance)'
)

# ── t-SNE ─────────────────────────────────────────────────────────────────────
print("\nRunning t-SNE (PCA 50 → t-SNE 2)...")
pca50   = PCA(n_components=50, random_state=42)
X_pca50 = pca50.fit_transform(X_flat)

tsne   = TSNE(n_components=2, random_state=42, perplexity=30,
              max_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_pca50)

plot_embedding(
    X_tsne, y,
    title='t-SNE — Multivariate Dataset',
    filename='tsne_rus.png',   # ← changed
    xlabel='t-SNE Dimension 1',
    ylabel='t-SNE Dimension 2'
)

print("\nAll done!")

### SMOTE for oversampling (only train set) and shuffling

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.utils import shuffle
from collections import Counter
import numpy as np
import pickle
import pandas as pd
import os

# 1. SETUP: Folders and Data Preparation
input_dir = './mvts_rus'
output_dir = './SMOTEData'
os.makedirs(output_dir, exist_ok=True)

# SMOTE requires 2D input (samples, features)
# We flatten 3D (samples, 288, 10) -> 2D (samples, 2880) [cite: 757, 1131]
with open(f'{input_dir}/X.pkl', 'rb') as f:
    X_train = pickle.load(f)

y_train = pd.read_csv(f'{input_dir}/y.csv', header=None).values.flatten().astype(int)


X_train_flat = X_train.reshape(X_train.shape[0], -1)
print(f"Original Training Balance: {Counter(y_train)}")

# 2. EXECUTION: SMOTE Oversampling
# This generates synthetic SEP samples to match the NSEP count [cite: 1125, 1133]
smote = SMOTE(random_state=42)
X_resampled_flat, y_resampled = smote.fit_resample(X_train_flat, y_train)

# 3. EXECUTION: Random Shuffling
# Mixes the synthetic data into the real data so they aren't clumped at the end [cite: 1163]
X_train_final_flat, y_train_final = shuffle(X_resampled_flat, y_resampled, random_state=42)

# Reshape back to 3D (samples, 288, 10) for your model [cite: 1134]
X_train_final = X_train_final_flat.reshape(X_train_final_flat.shape[0], 288, 10)

# 4. STORAGE: Save to SMOTEData Folder
# Save the 3D features as a .pkl file [cite: 1172]
pkl_path = os.path.join(output_dir, 'X_train_smote.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(X_train_final, f)

# Save the labels as a separate 1D CSV file [cite: 1172]
csv_path = os.path.join(output_dir, 'y_train_smote.csv')
pd.Series(y_train_final).to_csv(csv_path, index=False, header=False)

print(f"\n--- SMOTE and Shuffling Complete ---")
print(f"Final Training Balance: {Counter(y_train_final)}")
print(f"Saved Features: {pkl_path}")
print(f"Saved Labels:   {csv_path}")

### SVM for classification (do hyperparameter tuning on evaluation set)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
import pickle
import pandas as pd
import os

# --- 1. Pull Data from Files ---
# Load Balanced Training Data from SMOTEData
with open('./SMOTEData/X_train_smote.pkl', 'rb') as f:
    X_train_final = pickle.load(f)
y_train_final = pd.read_csv('./SMOTEData/y_train_smote.csv', header=None).values.flatten()

# Load Original Validation Data from final_split_data
with open('./final_split_data/val_set.pkl', 'rb') as f:
    val_bundle = pickle.load(f)
X_val = val_bundle['X']
y_val = val_bundle['y']

# --- 2. Prepare Data for SVM ---
# SVM requires 2D input (samples, features)
X_train_2d = X_train_final.reshape(X_train_final.shape[0], -1)
X_val_2d   = X_val.reshape(X_val.shape[0], -1)

print(f"Data Loaded Successfully:")
print(f"  Training:   {X_train_2d.shape}")
print(f"  Validation: {X_val_2d.shape}")

# --- 3. Hyperparameter Tuning ---
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'sigmoid'] 
}

print("\nStarting Hyperparameter Tuning on Evaluation Set...")
svm_model = SVC(random_state=42)
# prioritization of 'recall' ensures we catch as many SEP events as possible
grid_search = GridSearchCV(svm_model, param_grid, cv=3, scoring='recall', verbose=2)
grid_search.fit(X_train_2d, y_train_final)

# --- 4. Save the Results ---
print(f"\nBest Settings Found: {grid_search.best_params_}")
best_svm = grid_search.best_estimator_

os.makedirs('./final_evaluation_ready', exist_ok=True)
with open('./final_evaluation_ready/best_svm_model.pkl', 'wb') as f:
    pickle.dump(best_svm, f)

### Evaluation Metrics (TSS, HSS1, HSS2, GSS, and Recall) 

In [ ]:
import pickle
import pandas as pd
import numpy as np
import os
from sklearn.metrics import confusion_matrix, f1_score

# 1. Load the Best Model and the Test Data
with open('./final_evaluation_ready/best_svm_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('./final_split_data/test_set.pkl', 'rb') as f:
    test_bundle = pickle.load(f)

X_test = test_bundle['X']
y_test = test_bundle['y']

# 2. Prepare and Predict
X_test_2d = X_test.reshape(X_test.shape[0], -1)
y_pred = model.predict(X_test_2d)

# 3. Calculate Confusion Matrix Elements
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# 4. Define Formula Functions
def calculate_metrics(tn, fp, fn, tp, y_true, y_pred):
    total = tn + fp + fn + tp
    
    # Accuracy
    accuracy = (tp + tn) / total if total > 0 else 0
    
    # Recall
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # F1-Score (using sklearn for precision/recall balance)
    f1 = f1_score(y_true, y_pred)
    
    # TSS
    tss = (tp / (tp + fn)) - (fp / (fp + tn)) if (tp + fn) > 0 and (fp + tn) > 0 else 0
    
    # HSS1
    hss1_num = (tp + tn) - ((tp + fn) * (tp + fp) + (tn + fn) * (tn + fp)) / total
    hss1_den = total - ((tp + fn) * (tp + fp) + (tn + fn) * (tn + fp)) / total
    hss1 = hss1_num / hss1_den if hss1_den != 0 else 0

    # HSS2
    hss2_num = 2 * (tp * tn - fp * fn)
    hss2_den = (tp + fn) * (fn + tn) + (tp + fp) * (fp + tn)
    hss2 = hss2_num / hss2_den if hss2_den != 0 else 0
    
    # GSS
    tp_random = ((tp + fn) * (tp + fp)) / total
    gss = (tp - tp_random) / (tp + fn + fp - tp_random) if (tp + fn + fp - tp_random) != 0 else 0
    
    return accuracy, recall, f1, tss, hss1, hss2, gss

# Calculate all metrics
accuracy, recall, f1, tss, hss1, hss2, gss = calculate_metrics(tn, fp, fn, tp, y_test, y_pred)

# 5. Save to .txt file in SavedResults folder
output_folder = './SavedResults'
os.makedirs(output_folder, exist_ok=True)
file_path = os.path.join(output_folder, 'SMOTEmetrics.txt')

# Format: TP, TN, FP, FN, TSS, HSS1, HSS2, GSS, Recall, Accuracy, F1-Score
metric_string = f"{tp},{tn},{fp},{fn},{tss:.4f},{hss1:.4f},{hss2:.4f},{gss:.4f},{recall:.4f},{accuracy:.4f},{f1:.4f}"

with open(file_path, 'w') as f:
    f.write(metric_string)

print("--- FINAL EVALUATION METRICS ---")
print(f"Total Test Samples: {len(y_test)}")
print(f"Confusion Matrix: [TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}]")
print("-" * 30)
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall:   {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"TSS:      {tss:.4f}")
print(f"HSS1:     {hss1:.4f}")
print(f"HSS2:     {hss2:.4f}")
print(f"GSS:      {gss:.4f}")
print(f"Saved results to: {file_path}")

# 5. Output Results
